# GC-SWGAN · Member 3 · Prompt 2: Evaluation and loss verification

**This notebook does not train the model or use the test set.** It adds reproducible plots/metrics, runs synthetic tests, and inspects the *actual* checked-out repository for loss-audit evidence.

**Before starting:** Kaggle internet may be enabled to clone the repository. Attach `member3_prompt2_package.zip` as an Input dataset via **Input → Upload → New Dataset**, then **Add Input**. Attach your Galaxy10 dataset only if you need it later; this notebook's tests do **not** need the 2.7 GB HDF5 file.

**Safety:** Your M3.1 audit found cross-split near-duplicate-looking galaxies and other sky-footprint overlaps. **The data is NOT cleared for full experiments.**

## Step 1 — Check repo and unpack these files into it
This notebook reuses `/kaggle/working/Research_Methodology` if it exists. The only files copied into the clone are Member 3 files, not changes to the Generator/trainer.

In [ ]:
import os, sys, shutil, subprocess, zipfile
from pathlib import Path
REPO=Path('/kaggle/working/Research_Methodology')
if not REPO.is_dir():
    subprocess.run(['git','clone','https://github.com/purveee04/Research_Methodology.git',str(REPO)], check=True)
print('Repo:',REPO)
print('Git commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())

required=['data_eval.py','test_data_eval.py','member3_source_audit.py']
for name in required:
    if (REPO/name).is_file(): continue
    found=[]
    for root,dirs,files in os.walk('/kaggle/input'):
        if name in files: found.append(Path(root)/name)
        for fn in files:
            if fn=='member3_prompt2_package.zip':
                z=Path(root)/fn
                with zipfile.ZipFile(z) as ar:
                    if name in ar.namelist():
                        (REPO/name).write_bytes(ar.read(name));break
        if (REPO/name).is_file():break
    if not (REPO/name).is_file():
        if len(found)==1: shutil.copy2(found[0],REPO/name)
        else: raise FileNotFoundError(f'Please attach member3_prompt2_package.zip; missing {name}; matches={found}')
os.chdir(REPO)
if str(REPO) not in sys.path:sys.path.insert(0,str(REPO))
os.environ.setdefault('TF_USE_LEGACY_KERAS','1')
OUT=Path('/kaggle/working/member3_eval_output');OUT.mkdir(exist_ok=True)
print('Member 3 files ready:',*[str(REPO/f) for f in required],sep='\n - ')

## Step 2 — Run Member 3 synthetic tests
These verify *our new evaluation functions*, not galaxy-model quality. **Expected:** 9 tests, `OK`. The cell raises an error if any test fails.

In [ ]:
result=subprocess.run([sys.executable,'-m','unittest','test_data_eval','-v'],cwd=REPO,text=True,capture_output=True)
print(result.stdout);print(result.stderr)
assert result.returncode==0, 'Evaluation tests FAILED; do not continue without investigating.'
print('PASS: Member 3 evaluation unit tests')

## Step 3 — Print source evidence for the existing losses and training pipeline
This reads the **actual Kaggle repo**. It identifies source locations to inspect, but **text matches are not proof** of correct gradient behavior.

In [ ]:
import json
from member3_source_audit import audit_sources
static_report=audit_sources(REPO)
p=(OUT/'source_audit.json');p.write_text(json.dumps(static_report,indent=2))
for topic,detail in static_report['checks'].items():
    print('\n',topic,detail['status'],detail.get('file',''))
    print('Lines:',detail.get('token_line_numbers',{}))
print('Saved:',p)

## Step 4 — Inspect *actual* implementations manually
Review extracted functions, signs and gradient paths. All checks remain **NEEDS REVIEW** until you inspect/test them. These keywords only locate code snippets.

In [ ]:
needles=['gradient_penalty','lambda_rec','reconstruction','pseudo','apply_gradients','full_metrics','predict_probs','skip_test']
for filename in ['training_module.py','train_gc_swgan_i2i.py']:
    p=REPO/filename
    if not p.exists(): print('Missing:',p);continue
    lines=p.read_text().splitlines()
    print('\n###',filename)
    hits=[i for i,line in enumerate(lines) if any(x in line.lower() for x in needles)]
    for i in hits[:24]:
        print(f'{i+1:4d}:',lines[i][:180])
    print('Total matching lines:',len(hits),'(showing at most 24)')

## Step 5 — Run existing TensorFlow tests (optional if time permits)
The team reported passing tests previously; this reruns them in **your** Kaggle environment. A failure is meaningful and must be investigated. The commands do not modify the code.

In [ ]:
# Set True to run the actual existing repo test modules. This can take several minutes.
RUN_TEAM_TESTS=False
if RUN_TEAM_TESTS:
    for mod in ['test_generator_module','test_training_module']:
        r=subprocess.run([sys.executable,'-m','unittest',mod,'-v'],cwd=REPO,text=True,capture_output=True,timeout=1800)
        (OUT/f'{mod}_test_output.txt').write_text(r.stdout+'\n'+r.stderr)
        print(mod,'exit code',r.returncode,'(output saved)')
        print('\n'.join((r.stdout+'\n'+r.stderr).splitlines()[-12:]))
        assert r.returncode==0, f'{mod} failed; inspect output text'
else: print('SKIPPED. Set RUN_TEAM_TESTS=True to run the existing TensorFlow tests.')

## Step 6 — Test evaluation functions on synthetic predictions
These are **fake examples**, not validation or test results from your galaxy model.

In [ ]:
import numpy as np
from data_eval import evaluate_probabilities,save_json,plot_confusion_matrix
true=np.arange(10)
prob=np.eye(10)*.91+.009
prob=prob/prob.sum(axis=1,keepdims=True)
prob[9]=prob[0]    # deliberately classify one sample incorrectly
m=evaluate_probabilities(true,prob)
print('Synthetic accuracy:',m['accuracy'])
print('Synthetic macro-F1 (all 10 classes):',m['macro_f1_all_10'])
print('Synthetic balanced accuracy:',m['balanced_accuracy'])
save_json(m,OUT/'synthetic_metrics.json')
plot_confusion_matrix(m['confusion_matrix'],OUT/'synthetic_confusion_matrix.png')
print('Created synthetic metrics + confusion matrix; DO NOT report as model accuracy.')

## Step 7 — Make synthetic log plots and A/B validation comparison
Later use this same code on actual `loss.csv`/`val.csv` files. This demo makes two tiny invented CSVs and labels them clearly.

In [ ]:
import pandas as pd
from data_eval import plot_log_curve,plot_pseudo_acceptance,compare_validation_runs,fair_config_diff
frames=OUT/'synthetic_csv';frames.mkdir(exist_ok=True)
a=frames/'A_val.csv';b=frames/'B_val.csv';loss=frames/'A_loss.csv'
pd.DataFrame({'step':[1,2,3],'val_acc':[.11,.13,.16],'val_macro_f1':[.02,.03,.08], 'val_bal_acc':[.1,.11,.14]}).to_csv(a,index=False)
pd.DataFrame({'step':[1,2,3],'val_acc':[.10,.14,.18],'val_macro_f1':[.02,.05,.09], 'val_bal_acc':[.1,.13,.17]}).to_csv(b,index=False)
pd.DataFrame({'step':[1,2,3],'pseudo_accepted':[0,1,3],'pseudo_considered':[20,20,20],'g_loss':[-.1,-.2,-.3]}).to_csv(loss,index=False)
plot_log_curve(a,'accuracy',OUT/'synthetic_val_accuracy.png')
plot_log_curve(loss,'generator_loss',OUT/'synthetic_generator_loss.png')
plot_pseudo_acceptance(loss,OUT/'synthetic_pseudo_acceptance.png')
summary=compare_validation_runs(a,b,OUT/'synthetic_ab_comparison')
print('SYNTHETIC only:',summary)
print('Config check:',fair_config_diff({'seed':16,'experiment':'A'}, {'seed':16,'experiment':'B'}))

## Step 8 — OPTIONAL: plot real *validation* logs after completed runs
Do **not** use test metrics here, and do not invent CSV paths. Only enable after your team has completed matched runs using a leakage-approved dataset split. Check headers first; the code raises an error if columns cannot be recognized.

In [ ]:
# DO NOT enable until the team approves the data split and runs A/B.
USE_REAL_VAL_LOGS=False
VAL_A_CSV=''  # e.g. '/kaggle/working/.../A/val.csv'
VAL_B_CSV=''  # e.g. '/kaggle/working/.../B/val.csv'
LOSS_A_CSV='' # e.g. '/kaggle/working/.../A/loss.csv'
if USE_REAL_VAL_LOGS:
    assert VAL_A_CSV and VAL_B_CSV and LOSS_A_CSV, 'Set all three real paths'
    for name,path in [('A val',VAL_A_CSV),('B val',VAL_B_CSV),('A loss',LOSS_A_CSV)]:
        print(name, pd.read_csv(path,nrows=2).columns.tolist())
    compare_validation_runs(VAL_A_CSV,VAL_B_CSV,OUT/'real_ab_comparison')
    plot_pseudo_acceptance(LOSS_A_CSV,OUT/'real_A_pseudo_acceptance.png')
else: print('SKIPPED real metrics. Leakage not resolved; no reportable A/B results.')

## Step 9 — Save and download results
This ZIP contains your metric unit-test outputs, source audit, synthetic plots, and (if you enabled them) existing-team test logs. **It does not imply losses were verified or the split approved.**

In [ ]:
import shutil
zip_path=shutil.make_archive('/kaggle/working/member3_prompt2_results','zip',OUT)
print('Results ZIP:',zip_path)
print('Size MB:',round(Path(zip_path).stat().st_size/1e6,2))
from IPython.display import FileLink,display
display(FileLink(zip_path))

## What remains (do not skip)

1. Review the actual loss implementation and log **PASS / FAIL / NEEDS REVIEW** with source lines and runtime tests. The static audit is *navigation*, not certification.
2. Correct/approve the M3.1/M3.2 group-aware split; keep external cutouts excluded until verified.
3. Align macro-F1 definition with the existing `full_metrics()` implementation.
4. Obtain team-approved sigma, pseudo threshold, reconstruction weight, resolution, and matched run settings.
5. Run real A/B experiments from equivalent initial conditions, with separate saved checkpoints/logs. Tune only on validation, not test.
6. Create your own branch and PR after integration review. No file here modifies `main`.
